In [40]:
import os
import random
import json
import numpy as np
import soundfile as sf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from praatio import textgrid
from gudhi.point_cloud import timedelay
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go

# =============================================================================
# CONFIG
# =============================================================================

TEXTGRID_FILE = "../additional/data/ALL_049_F_ENG_ENG_HT1.Textgrid"
WAV_FILE = "../additional/data/ALL_049_F_ENG_ENG_HT1.wav"

DESKTOP = Path.home() / "Desktop"
OUTPUT_DIR = DESKTOP / "phone_cloud_images"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = OUTPUT_DIR / "manifest.json"

VOWELS = ['ɔj','ɛ','ə','ɪ','aj','ɑ','æ','i','o','ʊ','aw','e','u','a']
CONSONANTS = ['b','f','m','ɹ','ð','w','h','p','t','z','n','ɡ','dʒ','s','ʃ','v','l','ŋ','k','θ','j','tʃ','ʒ','d']
ALL_PHONES = VOWELS + CONSONANTS

M = 100
HEAD_TAIL_THRESHOLD = 0.03
MIN_LENGTH = 500

# Original bright Squid Game pink-red
SQUID_PINK_RED = (237, 27, 118)

random.seed(42)

# =============================================================================
# HELPERS
# =============================================================================

def wav_fraction_finder(sig, samplerate, start_time, end_time):
    return sig[int(start_time * samplerate):int(end_time * samplerate)]

def head_tail_scissor(sig, threshold=HEAD_TAIL_THRESHOLD, min_len=MIN_LENGTH):
    valid_interval = [idx for idx in range(len(sig)) if sig[idx] > threshold]
    if len(valid_interval) == 0:
        return False, sig
    head, tail = min(valid_interval), max(valid_interval)
    sig = sig[head:tail + 1]
    if tail - head < min_len:
        return False, sig
    return True, sig

def crop_to_content(pil_img, padding=3, bg_color=SQUID_PINK_RED, threshold=50):
    """Crop to minimal margins, treating bg_color as background."""
    arr = np.array(pil_img.convert('RGB'))
    diff = arr.astype(float) - np.array(bg_color)
    dist = np.sqrt(np.sum(diff**2, axis=2))
    mask = dist > threshold
    if not mask.any():
        return pil_img
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    rmin = max(0, rmin - padding)
    rmax = min(arr.shape[0]-1, rmax + padding)
    cmin = max(0, cmin - padding)
    cmax = min(arr.shape[1]-1, cmax + padding)
    return pil_img.crop((cmin, rmin, cmax+1, rmax+1))

# =============================================================================
# MAIN
# =============================================================================

print("=" * 60)
print("PART 1: Generating 2 random instances per phone...")
print("=" * 60)

sig, samplerate = sf.read(WAV_FILE)
manifest = []

for phone in ALL_PHONES:
    tg = textgrid.openTextgrid(TEXTGRID_FILE, includeEmptyIntervals=False)
    phoneTier = tg.getTier('phones')
    
    entries = [ele for ele in phoneTier.entries if ele[2] == phone]
    valid_list = [
        head_tail_scissor(wav_fraction_finder(sig, samplerate, e[0], e[1]))[1]
        for e in entries
        if head_tail_scissor(wav_fraction_finder(sig, samplerate, e[0], e[1]))[0]
    ]
    count = len(valid_list)
    
    if count == 0:
        print(f"  /{phone}/ — 0 instances, skipped")
        continue
    
    n_pick = min(2, count)
    picked_indices = random.sample(range(count), n_pick)
    
    for inst_idx, element in enumerate(picked_indices):
        data = valid_list[element]
        
        # Part 2 (silent)
        fig2, ax2 = plt.subplots(figsize=(4, 2))
        ax2.plot(data)
        plt.close(fig2)
        
        # Part 3 (silent)
        point_Cloud = timedelay.TimeDelayEmbedding(M, 1, 1)
        Points = point_Cloud(data)
        X = StandardScaler().fit_transform(Points)
        pca = PCA(n_components=3, whiten=True)
        X_PCA = pca.fit_transform(X)
        
        fig3 = plt.figure(figsize=(4, 3))
        ax3 = fig3.add_subplot(1, 1, 1, projection='3d')
        ax3.plot3D(X_PCA[:,0], X_PCA[:,1], X_PCA[:,2])
        plt.close(fig3)
        
        # Part 4: Plotly with ORIGINAL bright pink-red background
        fig4 = go.Figure(data=go.Scatter3d(
            x=X_PCA[:,0], y=X_PCA[:,1], z=X_PCA[:,2],
            marker=dict(size=4, color=X_PCA[:,2], colorscale='Viridis', showscale=False),
            mode='markers',
        ))
        fig4.update_layout(
            width=600, height=600,
            paper_bgcolor='rgb(237,27,118)',
            plot_bgcolor='rgb(237,27,118)',
            autosize=False,
            showlegend=False,
            margin=dict(l=0, r=0, t=0, b=0),
            scene=dict(
                xaxis_visible=False, yaxis_visible=False, zaxis_visible=False,
                xaxis_showspikes=False, yaxis_showspikes=False, zaxis_showspikes=False,
                bgcolor='rgb(237,27,118)',
                camera=dict(up=dict(x=0,y=0,z=1), eye=dict(x=1.2,y=1.2,z=1.2)),
                aspectratio=dict(x=1,y=1,z=0.8),
                aspectmode='manual',
            ),
        )
        
        raw_png = OUTPUT_DIR / f"{phone}_{inst_idx}_raw.png"
        fig4.write_image(str(raw_png), scale=2)
        
        cropped = crop_to_content(Image.open(raw_png))
        final_png = OUTPUT_DIR / f"{phone}_{inst_idx}.png"
        cropped.save(final_png)
        raw_png.unlink()
        
        manifest.append({
            "phone": phone,
            "weight": count,
            "path": str(final_png),
            "type": "vowel" if phone in VOWELS else "consonant",
            "instance": inst_idx,
        })
    
    print(f"  /{phone}/ — {count:3d} instances → saved {n_pick} image(s)")

with open(MANIFEST_PATH, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"\nDone. Saved {len(manifest)} images to: {OUTPUT_DIR}")
print(f"Manifest: {MANIFEST_PATH}")

PART 1: Generating 2 random instances per phone...
  /ɔj/ —   2 instances → saved 2 image(s)
  /ɛ/ —  24 instances → saved 2 image(s)
  /ə/ —  27 instances → saved 2 image(s)
  /ɪ/ —  43 instances → saved 2 image(s)
  /aj/ —  10 instances → saved 2 image(s)
  /ɑ/ —  22 instances → saved 2 image(s)
  /æ/ —  17 instances → saved 2 image(s)
  /i/ —  18 instances → saved 2 image(s)
  /o/ — 0 instances, skipped
  /ʊ/ —   5 instances → saved 2 image(s)
  /aw/ —   7 instances → saved 2 image(s)
  /e/ — 0 instances, skipped
  /u/ — 0 instances, skipped
  /a/ — 0 instances, skipped
  /b/ —   5 instances → saved 2 image(s)
  /f/ —   2 instances → saved 2 image(s)
  /m/ —  23 instances → saved 2 image(s)
  /ɹ/ —  24 instances → saved 2 image(s)
  /ð/ —   3 instances → saved 2 image(s)
  /w/ —   6 instances → saved 2 image(s)
  /h/ —  10 instances → saved 2 image(s)
  /p/ — 0 instances, skipped
  /t/ —   9 instances → saved 2 image(s)
  /z/ —  26 instances → saved 2 image(s)
  /n/ —  23 instances 

In [3]:
import os
import random
import json
import numpy as np
from pathlib import Path
from PIL import Image, ImageDraw
from scipy import ndimage

# =============================================================================
# CONFIG
# =============================================================================

DESKTOP = Path.home() / "Desktop"
INPUT_DIR = DESKTOP / "phone_cloud_images"
MANIFEST_PATH = INPUT_DIR / "manifest.json"
EAR_REF = DESKTOP / "ear.jpeg"          # <-- changed from ear.jpg
FINAL_CLOUD_PATH = DESKTOP / "ear_phone_cloud.png"

MASK_SHAPE = (1200, 675)
BASE_SIZE_RANGE = (22, 160)
MIN_SIZE = 12
OVERLAP_FRACTION = 0.25
MIN_MASK_COVERAGE = 0.5

SQUID_PINK_RED = (237, 27, 118, 255)

random.seed(42)

# =============================================================================
# HELPERS
# =============================================================================

def create_ear_mask(width, height, reference_path=None):
    if reference_path and Path(reference_path).exists():
        ref = Image.open(reference_path).convert('L').resize((width, height), Image.LANCZOS)
        return np.array(ref) < 128
    w, h = width, height
    img = Image.new('L', (width, height), 0)
    draw = ImageDraw.Draw(img)
    outer = [(0.48*w,0.05*h),(0.22*w,0.12*h),(0.08*w,0.30*h),(0.03*w,0.52*h),
             (0.07*w,0.75*h),(0.25*w,0.92*h),(0.52*w,0.97*h),(0.78*w,0.88*h),
             (0.92*w,0.65*h),(0.88*w,0.38*h),(0.70*w,0.18*h)]
    draw.polygon(outer, fill=255)
    inner = [(0.48*w,0.22*h),(0.62*w,0.32*h),(0.58*w,0.55*h),(0.42*w,0.52*h),(0.35*w,0.35*h)]
    draw.polygon(inner, fill=0)
    tragus = [(0.75*w,0.55*h),(0.88*w,0.58*h),(0.85*w,0.68*h),(0.72*w,0.62*h)]
    draw.polygon(tragus, fill=255)
    return np.array(img) > 128

def draw_ear_background(width, height, reference_path=None, opacity=8):
    """Very subtle dark ear silhouette underneath for gentle depth."""
    overlay = Image.new('RGBA', (width, height), (0, 0, 0, 0))
    if reference_path and Path(reference_path).exists():
        ref = Image.open(reference_path).convert('L').resize((width, height), Image.LANCZOS)
        ref_arr = np.array(ref)
        is_ear = ref_arr < 128
        silhouette = np.zeros((height, width, 4), dtype=np.uint8)
        silhouette[is_ear] = [30, 30, 30, opacity]
        overlay = Image.fromarray(silhouette, 'RGBA')
    return overlay

def draw_ear_outline_on_top(width, height, reference_path=None, opacity=100, linewidth=2):
    """
    Softer dark ear contour — visible enough to read the shape,
    but thin and translucent so it doesn't eclipse the phone icons.
    """
    overlay = Image.new('RGBA', (width, height), (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay)
    
    if reference_path and Path(reference_path).exists():
        ref = Image.open(reference_path).convert('L').resize((width, height), Image.LANCZOS)
        ref_arr = np.array(ref)
        is_ear = ref_arr < 128
        
        dilated = ndimage.binary_dilation(is_ear, iterations=linewidth)
        edge = dilated ^ is_ear
        
        edge_y, edge_x = np.where(edge)
        for x, y in zip(edge_x, edge_y):
            # Softer dark gray, lower opacity — visible but not dominant
            draw.point((int(x), int(y)), fill=(45, 45, 45, opacity))
    else:
        w, h = width, height
        outer = [(0.48*w,0.05*h),(0.22*w,0.12*h),(0.08*w,0.30*h),(0.03*w,0.52*h),
                 (0.07*w,0.75*h),(0.25*w,0.92*h),(0.52*w,0.97*h),(0.78*w,0.88*h),
                 (0.92*w,0.65*h),(0.88*w,0.38*h),(0.70*w,0.18*h)]
        draw.polygon(outer, outline=(45,45,45,opacity), fill=(0,0,0,0))
    
    return overlay

def draw_gru_framework(width, height, opacity=60):
    overlay = Image.new('RGBA', (width, height), (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay)
    
    line_color = (255, 255, 255, opacity)
    node_color = (255, 255, 255, opacity + 20)
    
    def node(x, y, r=3):
        draw.ellipse([x-r, y-r, x+r, y+r], fill=node_color)
    
    def line(x1, y1, x2, y2, w=1):
        draw.line([(x1, y1), (x2, y2)], fill=line_color, width=w)
    
    cell_w, cell_h = 105, 120
    rows = int(height / cell_h) + 2
    cols = int(width / cell_w) + 2
    
    for row in range(rows):
        for col in range(cols):
            ox = col * cell_w + (row % 2) * (cell_w // 2) - 20
            oy = row * cell_h - 15
            
            if ox < -40 or oy < -40 or ox > width + 40 or oy > height + 40:
                continue
            
            nodes = {
                'x': (ox + 7, oy + 25),
                'h_prev': (ox + 7, oy + 80),
                'z': (ox + 40, oy + 20),
                'r': (ox + 40, oy + 55),
                'h_tilde': (ox + 40, oy + 88),
                'h_out': (ox + 75, oy + 55)
            }
            
            connections = [
                (nodes['x'], nodes['z']), (nodes['x'], nodes['r']), (nodes['x'], nodes['h_tilde']),
                (nodes['h_prev'], nodes['z']), (nodes['h_prev'], nodes['r']), (nodes['h_prev'], nodes['h_tilde']),
                (nodes['z'], nodes['h_out']), (nodes['r'], nodes['h_tilde']), (nodes['h_tilde'], nodes['h_out'])
            ]
            
            for (x1, y1), (x2, y2) in connections:
                line(x1, y1, x2, y2, w=1)
            
            for name, (nx, ny) in nodes.items():
                node(nx, ny)
            
            next_ox = ox + cell_w
            if next_ox < width:
                next_hprev_x = next_ox + 7
                next_hprev_y = oy + 80
                mid_x = (nodes['h_out'][0] + next_hprev_x) // 2
                mid_y = (nodes['h_out'][1] + next_hprev_y) // 2 - 20
                draw.line([
                    (nodes['h_out'][0], nodes['h_out'][1]),
                    (mid_x, mid_y),
                    (next_hprev_x, next_hprev_y)
                ], fill=line_color, width=1)
            
            if row < rows - 1:
                below_ox = ox + ((row + 1) % 2) * (cell_w // 2) - 20
                below_hprev_x = below_ox + 7
                below_hprev_y = oy + cell_h + 80
                if 0 < below_hprev_x < width:
                    mid_x2 = (nodes['h_out'][0] + below_hprev_x) // 2
                    mid_y2 = (nodes['h_out'][1] + below_hprev_y) // 2
                    draw.line([
                        (nodes['h_out'][0], nodes['h_out'][1]),
                        (mid_x2, mid_y2 + 15),
                        (below_hprev_x, below_hprev_y)
                    ], fill=(255, 255, 255, max(15, opacity - 15)), width=1)
    
    return overlay

def place_image_cloud(images, weights, labels, mask_shape, 
                      reference_ear_path=None,
                      base_size_range=BASE_SIZE_RANGE,
                      min_size=MIN_SIZE):
    w, h = mask_shape
    mask = create_ear_mask(w, h, reference_path=reference_ear_path)
    
    # 1. Bright pink-red background
    canvas = Image.new('RGBA', (w, h), SQUID_PINK_RED)
    
    # 2. Very subtle dark ear silhouette underneath
    ear_bg = draw_ear_background(w, h, reference_path=reference_ear_path, opacity=8)
    canvas = Image.alpha_composite(canvas, ear_bg)
    
    # 3. GRU framework
    gru_overlay = draw_gru_framework(w, h, opacity=60)
    canvas = Image.alpha_composite(canvas, gru_overlay)
    
    # Normalize weights
    weights_arr = np.array(weights, dtype=float)
    if weights_arr.max() > 0:
        weights_arr = weights_arr / weights_arr.max()
    base_sizes = (base_size_range[0] + weights_arr * (base_size_range[1] - base_size_range[0]))
    
    order = np.argsort(base_sizes)[::-1]
    placed = []
    shrink_log = []
    
    for idx in order:
        img = images[idx]
        target_size = int(base_sizes[idx])
        current_size = target_size
        placed_ok = False
        
        while current_size >= min_size and not placed_ok:
            img_r = img.resize((current_size, current_size), Image.LANCZOS)
            if img_r.mode != 'RGBA':
                img_r = img_r.convert('RGBA')
            
            for _ in range(3000):
                x = random.randint(0, w - current_size)
                y = random.randint(0, h - current_size)
                cx, cy = x + current_size//2, y + current_size//2
                
                if not mask[cy, cx]:
                    continue
                
                ys, xs = np.mgrid[y:y+current_size, x:x+current_size]
                if mask[ys, xs].sum() / (current_size * current_size) < MIN_MASK_COVERAGE:
                    continue
                
                overlap = False
                for (px, py, ps) in placed:
                    ix1, iy1 = max(x, px), max(y, py)
                    ix2, iy2 = min(x+current_size, px+ps), min(y+current_size, py+ps)
                    if ix2 > ix1 and iy2 > iy1:
                        inter = (ix2 - ix1) * (iy2 - iy1)
                        if inter > OVERLAP_FRACTION * current_size * current_size:
                            overlap = True
                            break
                if overlap:
                    continue
                
                canvas.paste(img_r, (x, y), img_r)
                placed.append((x, y, current_size))
                placed_ok = True
                break
            
            if not placed_ok:
                current_size = int(current_size * 0.9)
        
        if not placed_ok:
            current_size = min_size
            img_r = img.resize((current_size, current_size), Image.LANCZOS)
            if img_r.mode != 'RGBA':
                img_r = img_r.convert('RGBA')
            for _ in range(10000):
                x = random.randint(0, w - current_size)
                y = random.randint(0, h - current_size)
                cx, cy = x + current_size//2, y + current_size//2
                if mask[cy, cx]:
                    canvas.paste(img_r, (x, y), img_r)
                    placed.append((x, y, current_size))
                    placed_ok = True
                    break
        
        if placed_ok and current_size < target_size:
            shrink_log.append(f"  {labels[idx]}: {target_size}px → {current_size}px")
    
    # 4. SUBTLE DARK EAR OUTLINE ON TOP — thin and translucent
    ear_outline = draw_ear_outline_on_top(w, h, reference_path=reference_ear_path, 
                                          opacity=100, linewidth=2)
    canvas = Image.alpha_composite(canvas, ear_outline)
    
    if shrink_log:
        print("Some images were shrunk to fit:")
        for line in shrink_log:
            print(line)
    
    return canvas

# =============================================================================
# MAIN
# =============================================================================

print("=" * 60)
print("PART 2: Building 1200x675 ear-shaped icon cloud...")
print("=" * 60)

with open(MANIFEST_PATH, 'r') as f:
    manifest = json.load(f)

print(f"Loaded manifest with {len(manifest)} images")

images = []
weights = []
labels = []
for item in manifest:
    img_path = Path(item["path"])
    if not img_path.exists():
        img_path = INPUT_DIR / img_path.name
    images.append(Image.open(img_path))
    weights.append(item["weight"])
    labels.append(item["phone"])

ear_cloud = place_image_cloud(
    images, weights, labels,
    mask_shape=MASK_SHAPE,
    reference_ear_path=str(EAR_REF),
    base_size_range=BASE_SIZE_RANGE,
    min_size=MIN_SIZE
)

# Convert to RGB and save
ear_cloud_rgb = ear_cloud.convert('RGB')
ear_cloud_rgb.save(FINAL_CLOUD_PATH)

print(f"\nSaved ear cloud to: {FINAL_CLOUD_PATH}")
print(f"  Dimensions: {MASK_SHAPE} (RGB)")
print(f"  Background: Squid Game pink-red (#ed1b76)")
print(f"  Ear contour: Subtle dark gray outline (opacity 100, linewidth 2)")
print(f"  GRU framework: Prominent white lattice")
print(f"  Images packed: {len(images)} / {len(images)} (100%)")

PART 2: Building 1200x675 ear-shaped icon cloud...
Loaded manifest with 58 images
Some images were shrunk to fit:
  ɪ: 160px → 116px
  ɪ: 160px → 93px
  s: 111px → 89px
  s: 111px → 89px
  ə: 108px → 78px
  ə: 108px → 87px
  z: 105px → 84px
  z: 105px → 84px
  ɹ: 99px → 72px
  ɹ: 99px → 64px
  ɛ: 99px → 57px
  ɛ: 99px → 57px
  n: 95px → 54px
  n: 95px → 54px
  m: 95px → 48px
  m: 95px → 48px
  ɑ: 92px → 46px
  ɑ: 92px → 41px
  i: 79px → 36px
  i: 79px → 40px
  æ: 76px → 38px
  æ: 76px → 34px
  ʃ: 76px → 34px
  ʃ: 76px → 30px
  d: 57px → 25px
  d: 57px → 22px
  aj: 54px → 18px
  aj: 54px → 21px
  h: 54px → 18px
  h: 54px → 14px
  t: 50px → 15px
  t: 50px → 17px
  ŋ: 50px → 13px
  ŋ: 50px → 12px
  tʃ: 50px → 12px
  tʃ: 50px → 12px
  aw: 44px → 12px
  aw: 44px → 12px
  w: 41px → 12px
  w: 41px → 17px
  l: 41px → 12px
  l: 41px → 13px
  b: 38px → 12px
  b: 38px → 12px
  ʊ: 38px → 12px
  ʊ: 38px → 12px
  ɡ: 31px → 12px
  ɡ: 31px → 12px
  ð: 31px → 12px
  ð: 31px → 12px
  f: 28px → 12px
  f: